Clinic Apppointment Analysis

The dataset was found in kaggle.

Goals for this analysis:
1. Data Cleaning
2. EDA

In [29]:
import pandas as pd

In [30]:
df = pd.read_csv('messy_clinic_appointments.csv')
df.head()

,patient_id,patient_name,age,gender,appointment_date,booking_date,doctor,department,billing_amount,follow_up_required
0,1080,Tammy Williams,76,female,2026/02/26,2024/12/03,Christopher Graham,Cardiology,£425.8,1
1,1074,Megan Strickland,69,F,05/23/2025,12-Jun-2024,Brandon Lewis,Orthopedics,€344.26,Y
2,1067,Amanda Schroeder,79,M,30-Nov-2025,"August 05, 24",Deanna Edwards,Neurology,€203.34,Y
3,1072,Anthony Mcpherson,47,F,"May 18, 25",09/09/2024,Karen Parsons,General,Rs85.76,No
4,1092,Benjamin Brown,45,NaN,2026/03/07,08/17/2024,Andrea Hernandez,General,$84.44,1


In [31]:
df.tail()

,patient_id,patient_name,age,gender,appointment_date,booking_date,doctor,department,billing_amount,follow_up_required
995,1082,Thomas Roman,64,Female,"September 14, 25",2024/07/15,Robert Bass,General,Rs200.43,Yes
996,1008,Jack Cooper,67,F,"April 09, 25","June 20, 24",Christopher Smith,General,Rs244.49,No
997,1022,Christopher Brown,28,1,"February 13, 26",2024/06/13,Kimberly Johnson,Neurology,£154.48,N
998,1088,Michael Sullivan,65,F,2025/07/04,2024/04/15,Anthony Mccoy,Neurology,$434.95,0
999,1003,Perry Crawford,79,Male,05-Nov-2025,2024/07/22,Marvin Hicks,Cardiology,€102.23,Y


In [32]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   patient_id          1000 non-null   int64
 1   patient_name        1000 non-null   str  
 2   age                 1000 non-null   int64
 3   gender              950 non-null    str  
 4   appointment_date    1000 non-null   str  
 5   booking_date        1000 non-null   str  
 6   doctor              1000 non-null   str  
 7   department          1000 non-null   str  
 8   billing_amount      950 non-null    str  
 9   follow_up_required  1000 non-null   str  
dtypes: int64(2), str(8)
memory usage: 78.3 KB


First things first, cols that have incosistent values:
1. gender - **this col also has 50 missing values**
2. appointment_date
3. booking_date
4. billing_amount - **this col also has 50 missing values**
5. follow_up_required

In [33]:
# working on col gender
df['gender'].value_counts() 

gender
0         129
1         128
Male      123
Female    120
F         116
female    112
male      112
M         110
Name: count, dtype: int64

so the gender col has 8 values representing the gender, 4 vals for each male and female. However, 0 and 1 could be quite tricky to see if what is Male and Female represented.

Now I will check waht the names correlated to those in the 0 and 1 gender, to properly check 0 and 1 gender.

In [34]:
df[df['gender'].isin(['0','1'])].head()

,patient_id,patient_name,age,gender,appointment_date,booking_date,doctor,department,billing_amount,follow_up_required
5,1057,Joe Sharp,78,1,"September 26, 25",08/31/2025,Kimberly Fields,General,NaN,Yes
12,1057,Chris Rice,61,1,2025/07/31,"May 05, 24",Amy Olson,Neurology,Rs277.14,N
25,1073,Phillip Simpson,88,0,"June 27, 25",05/06/2025,Darren Brooks,General,$51.72,Y
34,1071,Michele Park,46,0,10-Jan-2026,01-Nov-2024,Robert Sparks,General,£396.22,No
37,1053,John Yang,37,1,"September 15, 25",2024/05/09,Anthony Jensen,Orthopedics,$442.77,0


While it is not guaranteed due to no other cols having a perfect correlation to determining the gender, the names of the patients will do. Having majority of female-like names, 0 is considered Female while 1 is for the male gender.

Now it is time to convert all the values of the gender col into one consistent way. I decided to use Female and Male so that it is easier when visualize in a graph.

In [35]:
gender_map = {'0': 'Female', '1': 'Male', 'F': 'Female', 'M': 'Male', 'female': 'Female', 'male': 'Male', 'Female': 'Female', 'Male': 'Male'}
df['gender'] = df['gender'].map(gender_map)
df.head()

,patient_id,patient_name,age,gender,appointment_date,booking_date,doctor,department,billing_amount,follow_up_required
0,1080,Tammy Williams,76,Female,2026/02/26,2024/12/03,Christopher Graham,Cardiology,£425.8,1
1,1074,Megan Strickland,69,Female,05/23/2025,12-Jun-2024,Brandon Lewis,Orthopedics,€344.26,Y
2,1067,Amanda Schroeder,79,Male,30-Nov-2025,"August 05, 24",Deanna Edwards,Neurology,€203.34,Y
3,1072,Anthony Mcpherson,47,Female,"May 18, 25",09/09/2024,Karen Parsons,General,Rs85.76,No
4,1092,Benjamin Brown,45,NaN,2026/03/07,08/17/2024,Andrea Hernandez,General,$84.44,1


Now that the gender column is not consistent, it is now time to fill in those that has Nan / Missing values. First I will look into those values and check if there are a kind of pattern to better fill those values.

In [36]:
missing_gender = df[df['gender'].isnull()]
missing_gender.head()

,patient_id,patient_name,age,gender,appointment_date,booking_date,doctor,department,billing_amount,follow_up_required
4,1092,Benjamin Brown,45,NaN,2026/03/07,08/17/2024,Andrea Hernandez,General,$84.44,1
36,1003,Katelyn Smith,19,NaN,"October 16, 25",09/21/2025,Robert Henry,Neurology,$481.44,0
43,1063,Edward Sanders,29,NaN,2025/09/16,"April 29, 25",Steven Bailey,Orthopedics,Rs479.77,0
64,1077,Jennifer Stark,34,NaN,06/29/2025,04-Dec-2024,Jason Williams,Neurology,Rs270.33,N
70,1095,Jessica Heath,86,NaN,2025/06/25,04/24/2024,Sherri Long,Cardiology,€61.37,Yes


In [37]:
df['gender'].value_counts(dropna=False)

gender
Female    477
Male      473
NaN        50
Name: count, dtype: int64

Since the there are no cols that could make it that I could determing or based the gender of those that have null values, and since it is a client-appointment data where gender is a important basic information, I will make it then the gender is unknown so that it is marked as something that is needed to be known by the database. This will also make it easier for data visualization.

In [38]:
df['gender'] = df['gender'].fillna('Unknown')
df.head()

,patient_id,patient_name,age,gender,appointment_date,booking_date,doctor,department,billing_amount,follow_up_required
0,1080,Tammy Williams,76,Female,2026/02/26,2024/12/03,Christopher Graham,Cardiology,£425.8,1
1,1074,Megan Strickland,69,Female,05/23/2025,12-Jun-2024,Brandon Lewis,Orthopedics,€344.26,Y
2,1067,Amanda Schroeder,79,Male,30-Nov-2025,"August 05, 24",Deanna Edwards,Neurology,€203.34,Y
3,1072,Anthony Mcpherson,47,Female,"May 18, 25",09/09/2024,Karen Parsons,General,Rs85.76,No
4,1092,Benjamin Brown,45,Unknown,2026/03/07,08/17/2024,Andrea Hernandez,General,$84.44,1


Now moving onto the appointment_data col. From my initial look, it has incositent format of the dates, and it no missing values. With that I can proceed to reformatting the dates into the the ISO 8601 standard date format (YYY-MM-DD).

In [39]:
# changing format of col appointment_date to ISO 8601 standard format
df['appointment_date'] = pd.to_datetime(df['appointment_date'], format='mixed')
df['appointment_date'] = df['appointment_date'].dt.strftime('%Y-%m-%d')
df.head()

,patient_id,patient_name,age,gender,appointment_date,booking_date,doctor,department,billing_amount,follow_up_required
0,1080,Tammy Williams,76,Female,2026-02-26,2024/12/03,Christopher Graham,Cardiology,£425.8,1
1,1074,Megan Strickland,69,Female,2025-05-23,12-Jun-2024,Brandon Lewis,Orthopedics,€344.26,Y
2,1067,Amanda Schroeder,79,Male,2025-11-30,"August 05, 24",Deanna Edwards,Neurology,€203.34,Y
3,1072,Anthony Mcpherson,47,Female,2025-05-18,09/09/2024,Karen Parsons,General,Rs85.76,No
4,1092,Benjamin Brown,45,Unknown,2026-03-07,08/17/2024,Andrea Hernandez,General,$84.44,1


In [40]:
# changing format of col booking_date to ISO 8601 standard format
df['booking_date'] = pd.to_datetime(df['booking_date'], format='mixed')
df['booking_date'] = df['booking_date'].dt.strftime('%Y-%m-%d')
df.head()

,patient_id,patient_name,age,gender,appointment_date,booking_date,doctor,department,billing_amount,follow_up_required
0,1080,Tammy Williams,76,Female,2026-02-26,2024-12-03,Christopher Graham,Cardiology,£425.8,1
1,1074,Megan Strickland,69,Female,2025-05-23,2024-06-12,Brandon Lewis,Orthopedics,€344.26,Y
2,1067,Amanda Schroeder,79,Male,2025-11-30,2024-08-05,Deanna Edwards,Neurology,€203.34,Y
3,1072,Anthony Mcpherson,47,Female,2025-05-18,2024-09-09,Karen Parsons,General,Rs85.76,No
4,1092,Benjamin Brown,45,Unknown,2026-03-07,2024-08-17,Andrea Hernandez,General,$84.44,1


In [41]:
df.tail()

,patient_id,patient_name,age,gender,appointment_date,booking_date,doctor,department,billing_amount,follow_up_required
995,1082,Thomas Roman,64,Female,2025-09-14,2024-07-15,Robert Bass,General,Rs200.43,Yes
996,1008,Jack Cooper,67,Female,2025-04-09,2024-06-20,Christopher Smith,General,Rs244.49,No
997,1022,Christopher Brown,28,Male,2026-02-13,2024-06-13,Kimberly Johnson,Neurology,£154.48,N
998,1088,Michael Sullivan,65,Female,2025-07-04,2024-04-15,Anthony Mccoy,Neurology,$434.95,0
999,1003,Perry Crawford,79,Male,2025-11-05,2024-07-22,Marvin Hicks,Cardiology,€102.23,Y


Moving onto billing amount, I noticed that it uses different currencies. Before I changed into the a consistent currency, I will decide what that currecny will be. I have two choices that come in mind. First will be highest usage of currency seen in the dataframe, OR the currency first used by the first patient in the dataframe.

Reasoning for this for be:
Majority - it will be consistent based on the what most of the patient used as currency
First Patient - it could give a clue on where the clinic is located as the first patient mostly would come from locals.

I will need to better look into the data to see if the first-patient method could be feasible, if not I will go with the Majority method of currency format.

Additonally, the missing values would have to be remained null to preserve financial accountability, so that if ever patient information is needed for legal stuff like insurances, the databse keeps the billing amount true and be used as a proper way to check the patient info.

In [42]:
df.loc[[df['appointment_date'].idxmin(), df['booking_date'].idxmin()]]

,patient_id,patient_name,age,gender,appointment_date,booking_date,doctor,department,billing_amount,follow_up_required
127,1036,Daniel Ballard,38,Male,2025-03-26,2025-03-02,Willie Williams,Cardiology,£114.64,N
164,1082,Gary Walker,87,Unknown,2025-03-26,2024-03-26,Matthew Lopez,Orthopedics,Rs216.23,Yes


After checking both the appointment_date and booking_date for the earliest patient that booked and has been seen by the clinic. I deemed it to be too inconistent due to while patient have both appointment dates, it has different booking dates. This shows that the clinic could be running way longer with an older database. Additonally, both patient used different currency when paying does making it too incosistent.

With that analysis, I have decided with the majority method for the currency formatting. Now, I will first need to see all the currency that the dataframe has.

In [43]:
unique_currencies = df['billing_amount'].str.extract(r'([^\d.]+)')[0].unique()
unique_currencies

<StringArray>
['£', '€', 'Rs', '$', nan]
Length: 5, dtype: str

In [44]:
df_billing_amount = df[['billing_amount']].copy()

In [45]:
df_billing_amount['currency'] = df_billing_amount['billing_amount'].str.extract(r'([^\d.]+)').squeeze()
df_billing_amount['billing_amount'] = df_billing_amount['billing_amount'].str.replace(r'[^\d.]', '', regex=True).astype(float)
df_billing_amount.head()

,billing_amount,currency
0,425.80,£
1,344.26,€
2,203.34,€
3,85.76,Rs
4,84.44,$


Now that the currency has been extracted in the copy of the billing amount, it now time to convert it into the majority which I would now which has it.

In [46]:
df_billing_amount['currency'].value_counts()

currency
Rs    252
$     241
€     237
£     220
Name: count, dtype: int64

Rs or rupees are the majority of the currency. However it will be difficult to showcase a proper amount of the billing due to the conversion being to high. So with the I will go with the second majority which would be $. The conversion rate of the following would be:
* 1Rs = 0.010$
* 1€ = 1.14$
* 1£ = 1.34$

These are the conversion rate as of 2026/07/19

In [47]:
conversion_rate = {'Rs': 0.010, '€': 1.14, '£': 1.34}
rates = df_billing_amount['currency'].map(conversion_rate).fillna(1.0)
df_billing_amount['billing_amount'] *= rates
df_billing_amount['billing_amount'] = df_billing_amount['billing_amount'].round(2)

In [48]:
df_billing_amount.head()

,billing_amount,currency
0,570.57,£
1,392.46,€
2,231.81,€
3,0.86,Rs
4,84.44,$


Now it is time to add the converted billing amount from the df_billing_amount dataframe onto the original dataframe.

In [49]:
df['billing_amount_dollars'] = df_billing_amount['billing_amount']
df = df.drop(columns=['billing_amount'], errors='ignore')

In [50]:
df.head()

,patient_id,patient_name,age,gender,appointment_date,booking_date,doctor,department,follow_up_required,billing_amount_dollars
0,1080,Tammy Williams,76,Female,2026-02-26,2024-12-03,Christopher Graham,Cardiology,1,570.57
1,1074,Megan Strickland,69,Female,2025-05-23,2024-06-12,Brandon Lewis,Orthopedics,Y,392.46
2,1067,Amanda Schroeder,79,Male,2025-11-30,2024-08-05,Deanna Edwards,Neurology,Y,231.81
3,1072,Anthony Mcpherson,47,Female,2025-05-18,2024-09-09,Karen Parsons,General,No,0.86
4,1092,Benjamin Brown,45,Unknown,2026-03-07,2024-08-17,Andrea Hernandez,General,1,84.44


Keeping in mind that the billing amount col has Nan values on purpose for the sake of financial proof and legalities that may come when changing things in this col.

Now that the billing_amount_dollars col is now in one consistent currency, i will now move onto the last col needed for data cleaning: follow_up_required

First is that I noticed it also has different values representing 2 things, Yes and No. I will now check the values in this col.

In [51]:
df['follow_up_required'].value_counts()

follow_up_required
0      185
Y      182
Yes    181
N      154
1      151
No     147
Name: count, dtype: int64

In [52]:
yes_no_map = {'0':'No', '1':'Yes', 'No':'No', 'Yes':'Yes', 'N':'No', 'Y':'Yes'}
df['follow_up_required'] = df['follow_up_required'].map(yes_no_map)
df.head()

,patient_id,patient_name,age,gender,appointment_date,booking_date,doctor,department,follow_up_required,billing_amount_dollars
0,1080,Tammy Williams,76,Female,2026-02-26,2024-12-03,Christopher Graham,Cardiology,Yes,570.57
1,1074,Megan Strickland,69,Female,2025-05-23,2024-06-12,Brandon Lewis,Orthopedics,Yes,392.46
2,1067,Amanda Schroeder,79,Male,2025-11-30,2024-08-05,Deanna Edwards,Neurology,Yes,231.81
3,1072,Anthony Mcpherson,47,Female,2025-05-18,2024-09-09,Karen Parsons,General,No,0.86
4,1092,Benjamin Brown,45,Unknown,2026-03-07,2024-08-17,Andrea Hernandez,General,Yes,84.44


In [53]:
# one last check for of info from the dataset

In [54]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   patient_id              1000 non-null   int64  
 1   patient_name            1000 non-null   str    
 2   age                     1000 non-null   int64  
 3   gender                  1000 non-null   str    
 4   appointment_date        1000 non-null   str    
 5   booking_date            1000 non-null   str    
 6   doctor                  1000 non-null   str    
 7   department              1000 non-null   str    
 8   follow_up_required      1000 non-null   str    
 9   billing_amount_dollars  950 non-null    float64
dtypes: float64(1), int64(2), str(7)
memory usage: 78.3 KB


Now that the dataset is clean it is now time to import a new csv file with the cleaned version.

In [55]:
df.to_csv('cleaned_clinic_appointments.csv', index=False)